# 05 — WO-2 Analytical Investigation## PurposeThis notebook records the controlled execution and interpretation of the 21 canonical WO-2 SQL investigations. The work covers RAW catalog verification, assessment join fan-out, IMD missingness, registration and withdrawal patterns, assessment structure, due dates, scores, banked submissions, and participation coverage.## Implementation and evidence- Canonical SQL definitions are maintained in `sql/04_analytics/`.- The shared read-only PostgreSQL runner is maintained in `src/analytics/db.py`.- The notebook retains query outputs in `query_results` and displays each returned result set.- The saved execution record contains 21 canonical queries, 28 result sets, 857 displayed rows, and 34 executed code cells. No saved SQL traceback or runner row-cap truncation is recorded.- AN0.01 preflight and the AN1.01 dependency refresh are additional to the 21 canonical investigations.- The retained outputs are historical evidence from the local PostgreSQL session. This documentation review did not reconnect to the database or rerun SQL.## Execution controlsThe notebook uses a read-only database preflight and an explicit live-execution authorization gate. Query dependencies are protected by review checkpoints; a closed gate prevents SQL submission. AN3.06 is isolated because it scans the large `student_vle` table. Queries are not intended to run concurrently.RAW, QUALITY, and STAGING remain unchanged. The notebook creates no persistent database objects, indexes, or business-cleaning rules. Interpretations distinguish row grain, denominator, missingness, and time basis; descriptive associations are not treated as causal findings.The WO-1 evidence notebook remains `notebooks/04_sql_analytics.ipynb`; the separate Python exploratory notebook remains `notebooks/05_python_advanced_analytics.ipynb`. These historical filenames are retained rather than renamed.

## 1. Environment setup

This cell only locates the repository, imports the shared runner, and initializes the in-memory result dictionary. It does **not** connect to PostgreSQL. A separate following cell confirms that the helper and SQL inventory are available.

In [1]:
# 1. Resolve the repository root and import the shared read-only query runner.
from pathlib import Path
import sys
from IPython.display import display

_current = Path.cwd().resolve()
_candidates = [_current, *_current.parents]
PROJECT_ROOT = next(
    (
        candidate
        for candidate in _candidates
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root from the current Jupyter directory. "
        "Open Jupyter from the repository root or its notebooks folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}

print("Project root:", PROJECT_ROOT)
print("Canonical SQL folder:", PROJECT_ROOT / "sql" / "04_analytics")
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").is_dir())
print("Shared runner imported:", callable(run_query))

Project root: <project-root>
Canonical SQL folder: <project-root>\sql\04_analytics
Canonical SQL folder found: True
Shared runner imported: True


In [2]:
# 2. Confirm the notebook environment is ready before opening a database connection.
assert PROJECT_ROOT.is_dir(), "Project root is not available."
assert (PROJECT_ROOT / "sql" / "04_analytics").is_dir(), "Canonical SQL folder is missing."
assert callable(run_query), "The shared query runner was not imported."
assert isinstance(query_results, dict), "The query-results store was not initialized."

print("run_query defined:", "run_query" in globals())
print("query_results defined:", "query_results" in globals())
print("Existing results in memory:", len(query_results))
print("Environment validation: PASS")
print("Database connection: NOT YET OPENED")

run_query defined: True
query_results defined: True
Existing results in memory: 0
Environment validation: PASS
Database connection: NOT YET OPENED


**Environment checkpoint:** successful completion of the two preceding cells confirms only that the project files and Python runner are available. It does not prove database connectivity. The next preflight cell opens a separate read-only connection and verifies the actual database identity and session settings.

## 2. Database connection preflight — AN0.01

**Question:** Is the notebook connected to the intended PostgreSQL database and a read-only session?

**Expectation:** The database is `oulad_analytics`; both `transaction_read_only` and `default_transaction_read_only` are `on`.

**Grain:** One row per active database session.

**Source:** PostgreSQL session settings and system functions, through the existing canonical file `sql/04_analytics/AN0.01_database_identity.sql`.

**Safety:** READ-ONLY. The runner opens a separate connection with a read-only default, statement timeout, and lock timeout.

**Runtime:** Very low; metadata only.

**Read as:** Stop if the database name is unexpected or either read-only setting is not `on`. This is a fresh preflight, not a claim that the historical WO-1 output proves the current connection.

In [3]:
# 3. Open a read-only connection and verify the actual database/session identity.
try:
    preflight = run_query("AN0.01", max_rows=10, statement_timeout_ms=120000)
except Exception as exc:
    print("DATABASE PREFLIGHT: FAILED")
    print("Exception type:", type(exc).__name__)
    print("Exception message:", str(exc))
    raise

query_results["PREFLIGHT.AN0.01"] = preflight
print("Query file:", preflight["query_file"])
print("Elapsed seconds:", preflight["elapsed_seconds"])
print("Result sets:", preflight["result_set_count"])

for item in preflight["result_sets"]:
    print(
        f"Result set {item['statement_number']}: "
        f"{item['rows_displayed']} displayed row(s); "
        f"truncated={item['display_truncated']}"
    )
    display(item["data"])

if preflight["result_set_count"] != 1 or preflight["data"].empty:
    raise RuntimeError("Database preflight returned an unexpected result shape.")

row = preflight["data"].iloc[0]
checks = {
    "database_name": row["database_name"] == "oulad_analytics",
    "transaction_read_only": str(row["transaction_read_only"]).lower() == "on",
    "default_transaction_read_only": str(row["default_transaction_read_only"]).lower() == "on",
}
print("Preflight checks:", checks)

if not all(checks.values()):
    raise RuntimeError(
        "Database preflight failed. Stop here; do not execute WO-2 queries."
    )

PREFLIGHT_VERIFIED = True
print("DATABASE CONNECTION: VERIFIED")
print("READ-ONLY SESSION: VERIFIED")
print("WO-2 query execution may proceed one cell at a time.")

Query file: sql\04_analytics\AN0.01_database_identity.sql
Elapsed seconds: 0.009
Result sets: 1
Result set 1: 1 displayed row(s); truncated=False


,database_name,database_user,current_schema,server_version,transaction_read_only,default_transaction_read_only,statement_timeout,lock_timeout
0,oulad_analytics,postgres,public,18.6,on,on,2min,5s


Preflight checks: {'database_name': True, 'transaction_read_only': True, 'default_transaction_read_only': True}
DATABASE CONNECTION: VERIFIED
READ-ONLY SESSION: VERIFIED
WO-2 query execution may proceed one cell at a time.


### Preflight interpretation and decisionThe retained AN0.01 output identifies database `oulad_analytics` on PostgreSQL 18.6. Both `transaction_read_only` and `default_transaction_read_only` are `on`; the displayed statement timeout is 2 minutes and lock timeout is 5 seconds.**Decision:** the saved execution record passed the database-identity and read-only preflight. This is historical notebook evidence, not a new connection performed by this documentation update.

## 3. Controlled execution helper

The next code cell defines the authorization, query execution, dependency invalidation, and review-checkpoint functions. It does not open a database connection.

- A fresh kernel starts with live execution disabled.
- The helper requires the successful AN0.01 preflight and explicit live authorization.
- Each query reads exactly one canonical SQL file and stores all returned result sets.
- Review checkpoints verify output presence, result-set count, display completeness, and the SHA-256 hash of the SQL file used for execution.
- Expected workflow gates do not raise Python tracebacks. A closed gate prints why execution or review is blocked and returns safely.
- Actual SQL/database exceptions remain visible and are re-raised.
- The display cap is 500 rows per result set. A truncated display is never accepted at a review checkpoint.


In [4]:
# 4. Controlled query execution, explicit authorization, and safe review gates.
import hashlib

LIVE_WO2_EXECUTION_AUTHORIZED = False
PREFLIGHT_VERIFIED = globals().get("PREFLIGHT_VERIFIED", False)

# A fresh helper initialization starts all review gates closed.
AN0_07_REVIEWED = False
AN1_01_REVIEWED = False
AN1_10_REVIEWED = False
AN2_10_REVIEWED = False
AN3_REVIEWED = False
AN4_D_REVIEWED = False
AN4_06_REVIEWED = False

# Keep the fresh AN0.01 identity/read-only preflight, but discard any WO-2
# outputs left in memory if this helper cell is deliberately re-initialized.
for _result_key in list(query_results):
    if _result_key != "PREFLIGHT.AN0.01":
        query_results.pop(_result_key, None)

EXPECTED_RESULT_SET_COUNTS = {
    "AN0.07": 3,
    "AN1.10": 3,
    "AN2.10": 3,
    "AN3.01": 1,
    "AN3.02": 1,
    "AN3.03": 1,
    "AN3.04": 1,
    "AN3.05": 1,
    "AN3.06": 1,
    "AN3.07": 1,
    "AN3.08": 1,
    "AN4.01": 1,
    "AN4.02": 1,
    "AN4.04": 1,
    "AN4.05": 2,
    "AN4.06": 1,
    "AN4.03": 1,
    "AN4.07": 1,
    "AN4.08": 1,
    "AN4.09": 1,
    "AN4.10": 1,
}
assert len(EXPECTED_RESULT_SET_COUNTS) == 21

# Re-running an upstream query invalidates its own approval and every downstream
# approval/output that depends on it. This prevents stale results from opening gates.
DEPENDENCY_INVALIDATION = {
    "AN0.07": {
        "flags": [
            "AN0_07_REVIEWED", "AN1_01_REVIEWED", "AN1_10_REVIEWED",
            "AN2_10_REVIEWED", "AN3_REVIEWED", "AN4_D_REVIEWED",
            "AN4_06_REVIEWED",
        ],
        "results": [
            "PREFLIGHT.AN1.01", "AN1.10", "AN2.10",
            "AN3.01", "AN3.02", "AN3.03", "AN3.04",
            "AN3.05", "AN3.06", "AN3.07", "AN3.08",
            "AN4.01", "AN4.02", "AN4.04", "AN4.05",
            "AN4.06", "AN4.03", "AN4.07", "AN4.08",
            "AN4.09", "AN4.10",
        ],
    },
    "AN1.10": {
        "flags": [
            "AN1_10_REVIEWED", "AN2_10_REVIEWED", "AN3_REVIEWED",
            "AN4_D_REVIEWED", "AN4_06_REVIEWED",
        ],
        "results": [
            "AN2.10", "AN3.01", "AN3.02", "AN3.03", "AN3.04",
            "AN3.05", "AN3.06", "AN3.07", "AN3.08",
            "AN4.01", "AN4.02", "AN4.04", "AN4.05",
            "AN4.06", "AN4.03", "AN4.07", "AN4.08",
            "AN4.09", "AN4.10",
        ],
    },
    "AN2.10": {
        "flags": [
            "AN2_10_REVIEWED", "AN3_REVIEWED", "AN4_D_REVIEWED",
            "AN4_06_REVIEWED",
        ],
        "results": [
            "AN3.01", "AN3.02", "AN3.03", "AN3.04", "AN3.05",
            "AN3.06", "AN3.07", "AN3.08",
            "AN4.01", "AN4.02", "AN4.04", "AN4.05",
            "AN4.06", "AN4.03", "AN4.07", "AN4.08",
            "AN4.09", "AN4.10",
        ],
    },
}

AN3_QUERY_IDS = [
    "AN3.01", "AN3.02", "AN3.03", "AN3.04",
    "AN3.05", "AN3.06", "AN3.07", "AN3.08",
]
AN4_D_QUERY_IDS = ["AN4.01", "AN4.02", "AN4.04", "AN4.05"]
AN4_E_QUERY_IDS = ["AN4.06", "AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"]

for _query_id in AN3_QUERY_IDS:
    DEPENDENCY_INVALIDATION[_query_id] = {
        "flags": ["AN3_REVIEWED", "AN4_D_REVIEWED", "AN4_06_REVIEWED"],
        "results": AN4_D_QUERY_IDS + AN4_E_QUERY_IDS,
    }
for _query_id in AN4_D_QUERY_IDS:
    DEPENDENCY_INVALIDATION[_query_id] = {
        "flags": ["AN4_D_REVIEWED", "AN4_06_REVIEWED"],
        "results": AN4_E_QUERY_IDS,
    }
DEPENDENCY_INVALIDATION["AN4.06"] = {
    "flags": ["AN4_06_REVIEWED"],
    "results": ["AN4.03", "AN4.07", "AN4.08", "AN4.09", "AN4.10"],
}

def _canonical_sql_path(query_id):
    matches = sorted(
        (PROJECT_ROOT / "sql" / "04_analytics").glob(f"{query_id}_*.sql")
    )
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one canonical SQL file for {query_id}; "
            f"found {len(matches)}."
        )
    return matches[0]

def request_live_authorization():
    if not PREFLIGHT_VERIFIED:
        print("AUTHORIZATION BLOCKED: AN0.01 preflight has not passed.")
        return False
    answer = input(
        "Read-only preflight passed. Type AUTHORIZE to enable WO-2 queries "
        "for this kernel session: "
    )
    authorized = answer.strip().upper() == "AUTHORIZE"
    print("Live WO-2 execution authorized:", authorized)
    if not authorized:
        print("Execution remains disabled; no analytical SQL will run.")
    return authorized

def _invalidate_downstream(query_id):
    invalidation = DEPENDENCY_INVALIDATION.get(
        query_id, {"flags": [], "results": []}
    )
    for flag_name in invalidation["flags"]:
        globals()[flag_name] = False
    for stale_query_id in invalidation["results"]:
        query_results.pop(stale_query_id, None)

def _review_blockers(checkpoint_name, query_ids):
    blockers = []
    if not PREFLIGHT_VERIFIED:
        blockers.append("AN0.01 read-only database preflight is not verified.")
    if not LIVE_WO2_EXECUTION_AUTHORIZED:
        blockers.append("Live WO-2 execution has not been authorized.")

    for query_id in query_ids:
        result = query_results.get(query_id)
        if result is None:
            blockers.append(f"{query_id}: no returned output is stored in this kernel.")
            continue

        expected = EXPECTED_RESULT_SET_COUNTS[query_id]
        if result.get("result_set_count") != expected:
            blockers.append(
                f"{query_id}: expected {expected} result sets, "
                f"received {result.get('result_set_count')}."
            )
            continue
        result_sets = result.get("result_sets", [])
        if len(result_sets) != expected:
            blockers.append(f"{query_id}: result-set records are incomplete.")
            continue
        if any(item.get("display_truncated", False) for item in result_sets):
            blockers.append(
                f"{query_id}: at least one result set is truncated; review is blocked."
            )
        sql_path = _canonical_sql_path(query_id)
        current_hash = hashlib.sha256(sql_path.read_bytes()).hexdigest()
        if result.get("sql_sha256") != current_hash:
            blockers.append(
                f"{query_id}: canonical SQL changed after execution; rerun this query."
            )

    if checkpoint_name == "AN0.07" and not blockers:
        result_sets = query_results["AN0.07"]["result_sets"]
        key_data = result_sets[0]["data"]
        expected_tables = {
            "assessments", "courses", "student_assessment",
            "student_info", "student_registration", "student_vle", "vle",
        }
        if len(key_data) != 7 or set(key_data["table_name"]) != expected_tables:
            blockers.append("AN0.07: the seven-table RAW key inventory is incomplete.")
        else:
            statuses = key_data["key_status"].value_counts().to_dict()
            if statuses != {"MATCH": 6, "EXPECTED_NO_PK": 1}:
                blockers.append(
                    f"AN0.07: primary-key statuses require investigation: {statuses}"
                )
            elif not key_data.loc[
                key_data["key_status"] == "MATCH", "is_validated"
            ].all():
                blockers.append("AN0.07: at least one expected primary key is not validated.")

        constraint_data = result_sets[1]["data"]
        observed_constraints = dict(
            zip(
                constraint_data["constraint_type"],
                constraint_data["constraint_count"],
            )
        )
        expected_constraints = {"c": 0, "f": 0, "p": 6, "u": 0}
        if observed_constraints != expected_constraints:
            blockers.append(
                "AN0.07: constraint inventory differs from the expected reviewed "
                f"result: {observed_constraints}"
            )
        # The third result set is deliberately inspected by the human reviewer.
        # An empty index inventory is a finding, not an automatic failure.

    return blockers

def confirm_review(checkpoint_name, query_ids):
    blockers = _review_blockers(checkpoint_name, query_ids)
    if blockers:
        print(f"{checkpoint_name}: REVIEW BLOCKED")
        for blocker in blockers:
            print("-", blocker)
        print("No dependent query is authorized. Resolve the items above first.")
        return False

    print(
        f"{checkpoint_name}: outputs are present, complete, untruncated, "
        "and tied to the current canonical SQL."
    )
    answer = input(
        f"After the project lead has explicitly approved {checkpoint_name}, "
        "type REVIEWED to open its dependency gate: "
    )
    reviewed = answer.strip().upper() == "REVIEWED"
    print(f"{checkpoint_name} reviewed:", reviewed)
    if not reviewed:
        print(
            "REVIEW NOT RECORDED. The gate remains closed; this is not a SQL error. "
            "Rerun this checkpoint cell after approval."
        )
    return reviewed

def execute_query(query_id, *, statement_timeout_ms=120000, requires=None):
    if not LIVE_WO2_EXECUTION_AUTHORIZED:
        print(f"{query_id}: NOT EXECUTED — live WO-2 authorization is disabled.")
        return None
    if not PREFLIGHT_VERIFIED:
        print(f"{query_id}: NOT EXECUTED — AN0.01 preflight is not verified.")
        return None

    required_flags = [requires] if isinstance(requires, str) else (requires or [])
    missing_flags = [
        flag_name for flag_name in required_flags
        if not globals().get(flag_name, False)
    ]
    if missing_flags:
        print(f"{query_id}: NOT EXECUTED — review gate is still closed.")
        print("Required approval(s):", ", ".join(missing_flags))
        print("No SQL was sent to PostgreSQL. Wait for review approval, then retry.")
        return None

    _invalidate_downstream(query_id)
    query_results.pop(query_id, None)
    sql_path = _canonical_sql_path(query_id)
    sql_hash_before = hashlib.sha256(sql_path.read_bytes()).hexdigest()

    try:
        result = run_query(
            query_id,
            max_rows=500,
            statement_timeout_ms=statement_timeout_ms,
        )
    except Exception as exc:
        print(f"{query_id}: EXECUTION FAILED")
        print("Exception type:", type(exc).__name__)
        print("Exception message:", str(exc))
        raise

    sql_hash_after = hashlib.sha256(sql_path.read_bytes()).hexdigest()
    if sql_hash_after != sql_hash_before:
        raise RuntimeError(
            f"{query_id}: canonical SQL changed during execution. "
            "Discard this output and rerun after reviewing the file."
        )

    result["sql_sha256"] = sql_hash_before
    query_results[query_id] = result

    print(f"{query_id}: EXECUTION RETURNED")
    print("Query file:", result["query_file"])
    print("Result sets returned:", result["result_set_count"])
    print("Batch elapsed seconds:", result["elapsed_seconds"])
    print("Statement timeout (ms):", result["statement_timeout_ms"])
    print("Display row cap per result set:", result["max_rows"])

    for item in result["result_sets"]:
        status = "TRUNCATED" if item["display_truncated"] else "complete display"
        print(
            f"Result set {item['statement_number']}: "
            f"{item['rows_displayed']} row(s) displayed; {status}"
        )
        display(item["data"])

    expected = EXPECTED_RESULT_SET_COUNTS[query_id]
    if result["result_set_count"] != expected:
        raise RuntimeError(
            f"{query_id} returned {result['result_set_count']} result set(s); "
            f"{expected} were expected. Review the output and runner behavior."
        )

    print(f"{query_id}: RESULT-SET COUNT CHECK PASS ({expected})")
    return result


print("WO-2 helper initialized.")
print("AN0.01 preflight retained:", bool(PREFLIGHT_VERIFIED))
print("Previous WO-2 outputs cleared:", True)
print("Live execution authorization:", LIVE_WO2_EXECUTION_AUTHORIZED)


WO-2 helper initialized.
AN0.01 preflight retained: True
Previous WO-2 outputs cleared: True
Live execution authorization: False


### Explicit live-execution authorizationThe live-execution switch remains disabled by default. After the successful AN0.01 preflight, run the next cell and type `AUTHORIZE` at the prompt to enable read-only WO-2 queries for the current kernel session. No source-code edit is required. Any kernel restart resets authorization.


In [5]:
# Explicit authorization prompt — no source-code edit is required.
LIVE_WO2_EXECUTION_AUTHORIZED = request_live_authorization()

Read-only preflight passed. Type AUTHORIZE to enable WO-2 queries for this kernel session:  AUTHORIZE


Live WO-2 execution authorized: True


In [6]:
# 6. Confirm the execution helper, query inventory, and current safety gates.
assert callable(execute_query), "execute_query was not defined."
assert callable(confirm_review), "confirm_review was not defined."
assert isinstance(EXPECTED_RESULT_SET_COUNTS, dict)
assert len(EXPECTED_RESULT_SET_COUNTS) == 21
assert isinstance(query_results, dict)

print("execute_query defined:", callable(execute_query))
print("Expected WO-2 query IDs:", len(EXPECTED_RESULT_SET_COUNTS))
print("Result storage ready:", isinstance(query_results, dict))
print("Preflight verified:", PREFLIGHT_VERIFIED)
print("Live execution authorized:", LIVE_WO2_EXECUTION_AUTHORIZED)
print("WO-2 execution helper validation: PASS")


execute_query defined: True
Expected WO-2 query IDs: 21
Result storage ready: True
Preflight verified: True
Live execution authorized: True
WO-2 execution helper validation: PASS


## 4. WO2-A — Structural verification and join fan-out

The sequence is AN0.07 → review → current AN1.01 key refresh → review → AN1.10. The current AN1.01 refresh is a dependency check using the established canonical WO-1 SQL; it is not counted among the 21 WO-2 investigations.

### AN0.07 — RAW primary-key and index verification

**Question:** Do the primary-key constraints that actually exist in the RAW schema match the keys documented in `docs/data_dictionary.md`?

**Expectation:** Verify whether six RAW tables have validated primary keys on exactly the documented columns, in the documented order. Inspect the actual constraints and all indexes on `raw.student_vle` without assuming that the absence of a primary key implies the absence of indexes.

**Grain:** Statement 1: one row per RAW table; statement 2: one row per constraint type; statement 3: one row per index on `raw.student_vle`.

**Source:** PostgreSQL catalogs `pg_constraint`, `pg_class`, `pg_namespace`, `pg_attribute`, and `pg_indexes`.

**Safety:** READ-ONLY; catalog metadata only.

**Runtime:** Very low.

**Read as:** `key_status` must be `MATCH` for six tables and `EXPECTED_NO_PK` for `student_vle`. Any other value is an observation to investigate, not a reason to alter RAW. This check does not establish uniqueness of `student_vle` rows; that question is handled under AN5.

**Review gate:** Inspect all three result sets before proceeding.

In [7]:
an0_07 = execute_query("AN0.07", requires="PREFLIGHT_VERIFIED")

AN0.07: EXECUTION RETURNED
Query file: sql\04_analytics\AN0.07_raw_primary_key_verification.sql
Result sets returned: 3
Batch elapsed seconds: 0.159
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 7 row(s) displayed; complete display


,table_name,documented_key,constraint_name,actual_key,is_validated,key_status
0,assessments,id_assessment,pk_assessments,id_assessment,True,MATCH
1,courses,"code_module, code_presentation",pk_courses,"code_module, code_presentation",True,MATCH
2,student_assessment,"id_assessment, id_student",pk_student_assessment,"id_assessment, id_student",True,MATCH
3,student_info,"code_module, code_presentation, id_student",pk_student_info,"code_module, code_presentation, id_student",True,MATCH
4,student_registration,"code_module, code_presentation, id_student",pk_student_registration,"code_module, code_presentation, id_student",True,MATCH
5,student_vle,NaN,NaN,NaN,None,EXPECTED_NO_PK
6,vle,id_site,pk_vle,id_site,True,MATCH


Result set 2: 4 row(s) displayed; complete display


,constraint_type,constraint_count
0,c,0
1,f,0
2,p,6
3,u,0


Result set 3: 0 row(s) displayed; complete display


,schemaname,tablename,indexname,indexdef


AN0.07: RESULT-SET COUNT CHECK PASS (3)


### Interpretation and decision — AN0.07The saved catalog outputs list all seven RAW tables. Six documented primary keys match the actual ordered key columns; `raw.student_vle` has no primary key, as expected from its unresolved repeated-key pattern. The constraint inventory contains six primary keys and no unique, foreign-key, or check constraints. The `raw.student_vle` index inventory returned zero rows.**Decision:** the catalog evidence verifies declared constraints and index metadata only. It does not establish uniqueness of `student_vle` rows and does not authorize index creation or source modification.

In [8]:
# Run this checkpoint only after the project lead has reviewed and approved all three AN0.07 result sets.
AN0_07_REVIEWED = confirm_review("AN0.07", ["AN0.07"])


AN0.07: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved AN0.07, type REVIEWED to open its dependency gate:  REVIEWED


AN0.07 reviewed: True


### AN1.01 — Current student-presentation key refresh (dependency check)

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration` in the current database state?

**Expectation:** No NULL key components and no duplicate key groups in either table.

**Grain:** One row per source table.

**Source:** `staging.student_info` and `staging.student_registration`.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Both rows must report `PASS`, zero NULL key rows, and zero duplicate-key groups. This fresh check protects the grain assumption used by AN1.10.

In [9]:
# 5. Refresh the current student-presentation key dependency before AN1.10.
def run_an1_01_refresh():
    global AN1_01_REVIEWED

    if not LIVE_WO2_EXECUTION_AUTHORIZED:
        print("AN1.01 REFRESH: NOT EXECUTED — live WO-2 authorization is disabled.")
        return None
    if not PREFLIGHT_VERIFIED:
        print("AN1.01 REFRESH: NOT EXECUTED — AN0.01 preflight is not verified.")
        return None
    if not AN0_07_REVIEWED:
        print(
            "AN1.01 REFRESH: NOT EXECUTED — AN0.07 review gate is closed. "
            "No SQL was sent to PostgreSQL."
        )
        return None

    AN1_01_REVIEWED = False
    for flag_name in [
        "AN1_10_REVIEWED", "AN2_10_REVIEWED", "AN3_REVIEWED",
        "AN4_D_REVIEWED", "AN4_06_REVIEWED",
    ]:
        globals()[flag_name] = False
    for stale_query_id in [
        "AN1.10", "AN2.10", "AN3.01", "AN3.02", "AN3.03", "AN3.04",
        "AN3.05", "AN3.06", "AN3.07", "AN3.08", "AN4.01", "AN4.02",
        "AN4.04", "AN4.05", "AN4.06", "AN4.03", "AN4.07", "AN4.08",
        "AN4.09", "AN4.10",
    ]:
        query_results.pop(stale_query_id, None)

    sql_path = _canonical_sql_path("AN1.01")
    sql_hash_before = hashlib.sha256(sql_path.read_bytes()).hexdigest()
    try:
        result = run_query("AN1.01", max_rows=20, statement_timeout_ms=120000)
    except Exception as exc:
        print("AN1.01 REFRESH: EXECUTION FAILED")
        print("Exception type:", type(exc).__name__)
        print("Exception message:", str(exc))
        raise

    if hashlib.sha256(sql_path.read_bytes()).hexdigest() != sql_hash_before:
        print("AN1.01 REFRESH: BLOCKED — canonical SQL changed during execution.")
        return None

    result["sql_sha256"] = sql_hash_before
    query_results["PREFLIGHT.AN1.01"] = result

    print("Query file:", result["query_file"])
    print("Elapsed seconds:", result["elapsed_seconds"])
    print("Result sets:", result["result_set_count"])
    display(result["data"])

    if (
        result["result_set_count"] != 1
        or result["display_truncated"]
        or result["data"].empty
    ):
        print("AN1.01 DEPENDENCY: FAIL — result is incomplete or has an unexpected shape.")
        return None

    key_check = result["data"]
    expected_tables = {"student_info", "student_registration"}
    required_columns = {
        "table_name", "gate_status", "null_key_rows",
        "duplicate_key_groups", "excess_rows",
    }
    if (
        len(key_check) != 2
        or set(key_check["table_name"]) != expected_tables
        or not required_columns.issubset(key_check.columns)
    ):
        print("AN1.01 DEPENDENCY: FAIL — expected table rows or columns are missing.")
        return None

    passed = (
        key_check["gate_status"].astype(str).str.upper().eq("PASS").all()
        and key_check["null_key_rows"].eq(0).all()
        and key_check["duplicate_key_groups"].eq(0).all()
        and key_check["excess_rows"].eq(0).all()
    )
    if not passed:
        print("AN1.01 DEPENDENCY: FAIL — student-presentation key checks require investigation.")
        return None

    AN1_01_REVIEWED = True
    print(
        "AN1.01 current key-grain dependency: PASS "
        "(two expected tables; zero null, duplicate, or excess rows)."
    )
    return result

an1_01_refresh = run_an1_01_refresh()


Query file: sql\04_analytics\AN1.01_student_key_uniqueness.sql
Elapsed seconds: 0.245
Result sets: 1


,table_name,total_rows,null_key_rows,distinct_key_values,duplicate_key_groups,excess_rows,gate_status
0,student_info,32593,0,32593,0,0,PASS
1,student_registration,32593,0,32593,0,0,PASS


AN1.01 current key-grain dependency: PASS (two expected tables; zero null, duplicate, or excess rows).


### AN1.01 — Current student-presentation key refresh (dependency check)

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration` in the current database state?

**Expectation:** Exactly two rows are returned, one for each table; both have `gate_status = PASS`, zero NULL key rows, zero duplicate-key groups, and zero excess rows.

**Grain:** One row per student table.

**Source:** Canonical WO-1 query `AN1.01_student_key_uniqueness.sql`, executed as a fresh dependency check.

**Safety:** READ-ONLY. The shared runner enforces the read-only PostgreSQL connection.

**Runtime:** Low; scans the two student-presentation tables.

**Read as:** This refresh must pass before AN1.10 because the fan-out investigation depends on the current student-presentation grain. It is a dependency check, not an additional WO-2 query.


### AN1.10 — Assessment join fan-out decomposition

**Question:** Which students and presentations explain the 33,407-row increase in the `id_student`-only join from AN1.03?

**Expectation:** The excess equals `SUM(submissions * (presentations_of_student - 1))` and comes only from students registered in more than one presentation.

**Grain:** Statement 1: one row per presentation count of the student; statement 2: one row per module and presentation; statement 3: one reconciliation row.

**Source:** `staging.student_assessment`, `staging.assessments`, `staging.student_info`.

**Safety:** READ-ONLY.

**Runtime:** Low to moderate.

**Read as:** The fan-out is explained only if `reconciles = TRUE`, `submissions_without_student_info = 0`, and `unmatched_on_correct_path = 0`. A different total from the historical baseline is a finding to investigate, not an error. Never join `student_assessment` to `student_info` on `id_student` alone.

**Review gate:** Inspect all three result sets and require reconciliation before proceeding.

In [10]:
an1_10 = execute_query(
    "AN1.10",
    requires=["AN0_07_REVIEWED", "AN1_01_REVIEWED"],
)

AN1.10: EXECUTION RETURNED
Query file: sql\04_analytics\AN1.10_assessment_fanout_decomposition.sql
Result sets returned: 3
Batch elapsed seconds: 0.718
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 5 row(s) displayed; complete display


,presentation_count,submitting_students,submission_rows,naive_join_rows,excess_join_rows
0,1,20233,143080,143080,0
1,2,2917,28508,57016,28508
2,3,198,2081,6243,4162
3,4,20,235,940,705
4,5,1,8,40,32


Result set 2: 22 row(s) displayed; complete display


,code_module,code_presentation,submission_rows,submissions_by_repeat_students,pct_by_repeat_students
0,AAA,2013J,1633,75,4.59
1,AAA,2014J,1516,129,8.51
2,BBB,2013B,11256,400,3.55
3,BBB,2013J,14375,415,2.89
4,BBB,2014B,9993,566,5.66
5,BBB,2014J,7408,451,6.09
6,CCC,2014B,7489,3844,51.33
7,CCC,2014J,11451,6124,53.48
8,DDD,2013B,10373,2066,19.92
9,DDD,2013J,7936,2189,27.58


Result set 3: 1 row(s) displayed; complete display


,source_rows,naive_rows,observed_excess,explained_excess,reconciles,submissions_without_student_info,correct_path_rows,unmatched_on_correct_path,equals_historical_baseline
0,173912,207319,33407,33407,True,0,173912,0,True


AN1.10: RESULT-SET COUNT CHECK PASS (3)


### Interpretation and decision — AN1.10The unrestricted `id_student`-only join produces 207,319 rows against 173,912 source assessment rows, an excess of 33,407. The presentation-level decomposition attributes the multiplication to students with submissions across multiple presentations. The metadata-aware relationship path returns 173,912 rows with zero unmatched rows in the retained output.**Decision:** use the complete module/presentation relationship path for student-assessment analysis. The 33,407 excess is explained join fan-out, not additional source submissions.

In [11]:
AN1_10_REVIEWED = confirm_review("AN1.10", ["AN1.10"])


AN1.10: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved AN1.10, type REVIEWED to open its dependency gate:  REVIEWED


AN1.10 reviewed: True


## 5. WO2-B — IMD missingness and initial withdrawal investigation

### AN2.10 — IMD missingness extension

**Question:** Is missing `imd_band` concentrated by module or region-by-presentation, and can it be recovered from the same person in another presentation?

**Expectation:** Historical North Region (40.10%) and Ireland (22.47%) concentrations are hypotheses until checked. Some missing rows may have alternative values elsewhere.

**Grain:** Statement 1: module; statement 2: every region/module/presentation group with at least one missing IMD value; statement 3: recovery class (rows and distinct students).

**Source:** `staging.student_info`.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Reconcile missing rows to the historical 1,111-row reference. A value in another presentation does not prove the IMD of this presentation; conflicting values must not be resolved by picking one. No imputation is approved.

In [12]:
an2_10 = execute_query("AN2.10", requires="AN1_10_REVIEWED")

AN2.10: EXECUTION RETURNED
Query file: sql\04_analytics\AN2.10_imd_missingness_extension.sql
Result sets returned: 3
Batch elapsed seconds: 0.079
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 7 row(s) displayed; complete display


,code_module,student_presentation_rows,imd_missing_rows,pct_imd_missing
0,CCC,4434,254,5.73
1,FFF,7762,367,4.73
2,DDD,6272,280,4.46
3,EEE,2934,117,3.99
4,AAA,748,23,3.07
5,BBB,7909,63,0.80
6,GGG,2534,7,0.28


Result set 2: 95 row(s) displayed; complete display


,region,code_module,code_presentation,rows_total,imd_missing_rows,pct_imd_missing
0,Ireland,AAA,2013J,11,8,72.73
1,Ireland,CCC,2014J,87,42,48.28
2,Ireland,CCC,2014B,60,26,43.33
3,Ireland,EEE,2013J,33,13,39.39
4,Ireland,AAA,2014J,8,3,37.50
...,...,...,...,...,...,...
90,West Midlands Region,BBB,2014J,191,1,0.52
91,Yorkshire Region,EEE,2014B,44,1,2.27
92,Yorkshire Region,FFF,2013B,117,2,1.71
93,Yorkshire Region,CCC,2014B,114,1,0.88


Result set 3: 1 row(s) displayed; complete display


,recovery_class,missing_rows,distinct_students,pct_of_missing_rows,total_missing_rows
0,NO_ALTERNATIVE_VALUE,1111,971,100.00,1111


AN2.10: RESULT-SET COUNT CHECK PASS (3)


### Interpretation and decision — AN2.10The seven module totals reconcile to 32,593 student-presentation rows and 1,111 rows with missing `imd_band`: CCC 254, FFF 367, DDD 280, EEE 117, AAA 23, BBB 63, and GGG 7. The retained output identifies 971 distinct students among the 1,111 missing rows. The tested cross-presentation comparison classified all 1,111 rows as `NO_ALTERNATIVE_VALUE`; this does not establish the missingness mechanism or justify imputation.**Decision:** preserve NULLs and retain both row and distinct-student denominators. No IMD imputation rule is approved.

In [13]:
AN2_10_REVIEWED = confirm_review("AN2.10", ["AN2.10"])


AN2.10: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved AN2.10, type REVIEWED to open its dependency gate:  REVIEWED


AN2.10 reviewed: True


## 6. WO2-C — Outcomes, withdrawal, and registration timing

Execute AN3.01–AN3.08 individually. AN3.06 is isolated as a high-runtime scan and must not be combined with other queries.

### AN3.01 — Final result versus unregistration

**Question:** How consistent is final_result = 'Withdrawn' with the presence of date_unregistration?

**Expectation:** Historical counts (10,072 rows with an unregistration date and 10,156 Withdrawn rows) are reconciliation references only; the current cross-tab must be read without forcing those values.

**Grain:** One row per final_result and unregistration-date presence.

**Source:** staging.student_info joined to staging.student_registration on the full three-column key.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** The join is one-to-one (AN1.01). Non-zero off-diagonal cells are real data conflicts: do not define withdrawal from one field until they are explained.

**Execution note:** Inspect both off-diagonal populations; do not treat final_result and unregistration date as interchangeable.

In [14]:
an3_01 = execute_query(
    "AN3.01",
    requires="AN2_10_REVIEWED",
)

AN3.01: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.01_final_result_vs_unregistration.sql
Result sets returned: 1
Batch elapsed seconds: 0.046
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 6 row(s) displayed; complete display


,final_result,has_unregistration_date,student_presentation_rows,pct_within_final_result
0,Distinction,False,3024,100.00
1,Fail,False,7043,99.87
2,Fail,True,9,0.13
3,Pass,False,12361,100.00
4,Withdrawn,False,93,0.92
5,Withdrawn,True,10063,99.08


AN3.01: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.01The retained result contains 32,593 student-presentations. It records 10,156 `Withdrawn` outcomes: 10,063 have an unregistration date and 93 do not. Nine `Fail` rows have an unregistration date; all observed `Distinction` and `Pass` rows in this output have no such date.**Decision:** `final_result = 'Withdrawn'` and the presence of `date_unregistration` are related but not equivalent definitions. Keep both fields unchanged and do not collapse them into one withdrawal flag without a separate business decision.

### AN3.02 — Withdrawal definition conflicts

**Question:** Which rows break the equivalence Withdrawn <=> has date_unregistration, and how large is each conflict group by presentation?

**Expectation:** Potential conflict groups are Withdrawn without a date and non-Withdrawn with a date; their prevalence and timing must be established from the returned output.

**Grain:** One row per conflict group, module, and presentation.

**Source:** staging.student_info, staging.student_registration, staging.courses.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** NULL-safe comparison is used although final_result is NOT NULL in RAW. Do not label any group erroneous without context. Compare date_unregistration with module_presentation_length for the non-Withdrawn group. A date after the length suggests a post-course action rather than a withdrawal.

**Execution note:** Inspect conflict counts and timing relative to presentation length before defining any withdrawal rule.

In [15]:
an3_02 = execute_query(
    "AN3.02",
    requires="AN2_10_REVIEWED",
)

AN3.02: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.02_withdrawal_definition_conflicts.sql
Result sets returned: 1
Batch elapsed seconds: 0.058
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 15 row(s) displayed; complete display


,conflict_group,code_module,code_presentation,rows_in_group,min_unregistration_day,max_unregistration_day,after_course_end_rows
0,NOT_WITHDRAWN_WITH_DATE,BBB,2013J,3,0.0,166.0,0
1,NOT_WITHDRAWN_WITH_DATE,DDD,2013J,3,-7.0,0.0,0
2,NOT_WITHDRAWN_WITH_DATE,FFF,2013J,3,-4.0,0.0,0
3,WITHDRAWN_NO_DATE,BBB,2014B,1,NaN,NaN,0
4,WITHDRAWN_NO_DATE,BBB,2014J,13,NaN,NaN,0
5,WITHDRAWN_NO_DATE,CCC,2014J,28,NaN,NaN,0
6,WITHDRAWN_NO_DATE,DDD,2013B,1,NaN,NaN,0
7,WITHDRAWN_NO_DATE,DDD,2014B,1,NaN,NaN,0
8,WITHDRAWN_NO_DATE,DDD,2014J,16,NaN,NaN,0
9,WITHDRAWN_NO_DATE,EEE,2014J,4,NaN,NaN,0


AN3.02: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.02The 15 returned groups comprise 93 `WITHDRAWN_NO_DATE` rows and nine `NOT_WITHDRAWN_WITH_DATE` rows. Of the 93 Withdrawn rows without dates, 87 occur in 2014J (CCC 28, FFF 24, DDD 16, BBB 13, EEE 4, GGG 2); the remaining six are in other presentations. The nine non-Withdrawn rows with dates are all Fail in 2013J, three each in BBB, DDD, and FFF.**Decision:** preserve these as definition conflicts. The 2014J concentration is descriptive and does not establish a recording-process explanation.

### AN3.03 — Withdrawal timing distribution

**Question:** When during a presentation do students unregister?

**Expectation:** Historical timing patterns are hypotheses only; no bucket prevalence or module difference is assumed.

**Grain:** One row per module, presentation, and timing bucket.

**Source:** staging.student_registration with staging.courses.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Buckets are relative to presentation start (day 0). Negative days mean before start. Percentages use all unregistrations of that presentation as the denominator, not all registrations.

**Execution note:** The after-course-end condition takes precedence over in-course week buckets; inspect all timing categories.

In [16]:
an3_03 = execute_query(
    "AN3.03",
    requires="AN2_10_REVIEWED",
)

AN3.03: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.03_withdrawal_timing_distribution.sql
Result sets returned: 1
Batch elapsed seconds: 0.031
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 111 row(s) displayed; complete display


,code_module,code_presentation,timing_bucket,unregistration_rows,pct_of_presentation_unregistrations
0,AAA,2013J,1_before_start,7,11.67
1,AAA,2013J,2_weeks_1_4,6,10.00
2,AAA,2013J,3_weeks_5_12,12,20.00
3,AAA,2013J,4_weeks_13_24,16,26.67
4,AAA,2013J,5_weeks_25_to_end,19,31.67
...,...,...,...,...,...
106,GGG,2014J,1_before_start,20,16.13
107,GGG,2014J,2_weeks_1_4,12,9.68
108,GGG,2014J,3_weeks_5_12,44,35.48
109,GGG,2014J,4_weeks_13_24,37,29.84


AN3.03: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.03The saved query returned 111 grouped rows across module-presentations and timing buckets. Percentages use each presentation's recorded unregistration total, and the timing reference is the presentation start. The displayed pandas output abbreviates middle rows, so presentation-level extrema and complete bucket totals must be taken from the retained full result frame rather than inferred from the abbreviated display.**Decision:** retain the timing distribution as descriptive evidence. It does not identify students' reasons for withdrawal or establish a causal timing effect.

### AN3.04 — Registration timing profile

**Question:** How early do students register, and does registration timing relate to withdrawal?

**Expectation:** Historical observations (including 45 missing registration dates) are reference hypotheses only; verify current bucket counts and outcome rates.

**Grain:** One row per registration-timing bucket.

**Source:** staging.student_registration and staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Report the NULL bucket separately; never drop it silently. Association only: earlier registration may reflect motivation, not cause outcomes.

**Execution note:** Retain missing registration dates and interpret withdrawal differences as descriptive associations, not causal effects.

In [17]:
an3_04 = execute_query(
    "AN3.04",
    requires="AN2_10_REVIEWED",
)

AN3.04: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.04_registration_timing_profile.sql
Result sets returned: 1
Batch elapsed seconds: 0.039
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 6 row(s) displayed; complete display


,registration_bucket,student_presentation_rows,withdrawn_rows,pct_withdrawn
0,0_missing,45,39,86.67
1,1_more_than_180_days_before,795,376,47.30
2,2_91_to_180_days_before,8939,3354,37.52
3,3_31_to_90_days_before,13792,4091,29.66
4,4_1_to_30_days_before,8786,2247,25.57
5,5_on_or_after_start,236,49,20.76


AN3.04: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.04The six registration-timing groups cover all 32,593 student-presentations. The 45 missing registration dates include 39 Withdrawn outcomes (86.67%), a highly unstable proportion because the denominator is only 45. For dated registrations, observed Withdrawn shares are 47.30% (more than 180 days before; n=795), 37.52% (91–180 days; n=8,939), 29.66% (31–90 days; n=13,792), 25.57% (1–30 days; n=8,786), and 20.76% (on/after start; n=236).**Decision:** this is a descriptive association only. Previous attempts, education, presentation mix, and selection effects have not been controlled; do not interpret early registration as a cause of withdrawal.

### AN3.05 — Registration and unregistration order

**Question:** Are there impossible or edge-case date orderings (unregistered before registering, same day, before start)?

**Expectation:** No ordering pattern is assumed; historical counts are reference points only and must not determine classification.

**Grain:** One row per ordering category.

**Source:** staging.student_registration.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** BEFORE_REGISTRATION rows are data-quality candidates. SAME_DAY rows are legitimate but must be kept out of 'time enrolled' averages (zero duration).

**Execution note:** Classify missing, reversed, same-day, and later date sequences without changing source values.

In [18]:
an3_05 = execute_query(
    "AN3.05",
    requires="AN2_10_REVIEWED",
)

AN3.05: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.05_registration_unregistration_order.sql
Result sets returned: 1
Batch elapsed seconds: 0.061
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 4 row(s) displayed; complete display


,ordering_category,rows_in_category,min_days_enrolled,median_days_enrolled,max_days_enrolled
0,AFTER_REGISTRATION,10014,1.0,117.0,531.0
1,NO_UNREGISTRATION,22521,NaN,NaN,NaN
2,REGISTRATION_DATE_MISSING,39,NaN,NaN,NaN
3,SAME_DAY,19,0.0,0.0,0.0


AN3.05: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.05The four ordering categories reconcile to all 32,593 registration rows: 10,014 registrations with an unregistration date after registration, 22,521 without an unregistration date, 39 with a missing registration date, and 19 registered and unregistered on the same day. The 39 rows with missing registration dates and an unregistration date are not the same population as the 45 missing-registration-date rows in AN3.04.**Decision:** retain the distinct temporal and outcome definitions. Do not merge the 39 records into the 45 without checking the different conditions.

### AN3.06 — Withdrawn student-presentations and recorded VLE activity

**Question:** Among student-presentations classified as Withdrawn, how many have a recorded VLE activity row, and how does the last recorded activity day compare with the unregistration day?

**Expectation:** The query describes recorded-row presence and timing. No direction or prevalence is assumed.

**Grain:** One row per module, presentation, and recorded-activity group among Withdrawn student-presentations.

**Source:** staging.student_info, staging.student_registration, staging.student_vle (aggregated to student-presentation first).

**Safety:** READ-ONLY.

**Runtime:** HIGH. Scans staging.student_vle (historically about 10.6M rows). Run alone. Use the runner's explicit 600,000 ms timeout; do not add SET commands.

**Read as:** NO_ACTIVITY means no matching row was found in staging.student_vle for this student-presentation. It does not prove the student never clicked, did not learn, or used no other channel. Report both the Withdrawn denominator and the full student-presentation denominator. Do not use sum_click.

**Execution note:** Run this cell alone. Review the denominator, registration-match classification, and recorded-activity categories before proceeding.

In [19]:
an3_06 = execute_query(
    "AN3.06",
    statement_timeout_ms=600000,
    requires="AN2_10_REVIEWED",
)

AN3.06: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.06_withdrawn_activity_absence.sql
Result sets returned: 1
Batch elapsed seconds: 4.898
Statement timeout (ms): 600000
Display row cap per result set: 500
Result set 1: 76 row(s) displayed; complete display


,code_module,code_presentation,activity_group,withdrawn_student_presentations,distinct_students,withdrawn_denominator,all_student_presentation_denominator,pct_of_withdrawn_presentations,pct_of_all_student_presentations
0,AAA,2013J,LAST_ACTIVITY_AFTER_UNREGISTRATION,6,6,60,383,10.00,1.57
1,AAA,2013J,LAST_ACTIVITY_ON_OR_BEFORE_UNREGISTRATION,49,49,60,383,81.67,12.79
2,AAA,2013J,NO_RECORDED_ACTIVITY_ROW,5,5,60,383,8.33,1.31
3,AAA,2014J,LAST_ACTIVITY_AFTER_UNREGISTRATION,8,8,66,365,12.12,2.19
4,AAA,2014J,LAST_ACTIVITY_ON_OR_BEFORE_UNREGISTRATION,50,50,66,365,75.76,13.70
...,...,...,...,...,...,...,...,...,...
71,GGG,2014B,NO_RECORDED_ACTIVITY_ROW,34,34,100,833,34.00,4.08
72,GGG,2014J,ACTIVITY_UNREGISTRATION_DATE_MISSING,1,1,126,749,0.79,0.13
73,GGG,2014J,LAST_ACTIVITY_AFTER_UNREGISTRATION,4,4,126,749,3.17,0.53
74,GGG,2014J,LAST_ACTIVITY_ON_OR_BEFORE_UNREGISTRATION,88,88,126,749,69.84,11.75


AN3.06: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.06The saved output contains 76 module-presentation/activity-group rows for Withdrawn student-presentations. It distinguishes last recorded activity after unregistration, last activity on or before unregistration, no recorded activity row, and cases where the unregistration date is missing.**Decision:** treat this as a recorded-platform observation only. Missing activity is not proof of no learning or of no activity elsewhere; activity after unregistration does not prove that a student remained enrolled or participated in an accepted course activity.

### AN3.07 — Outcome rates by previous attempts and education

**Question:** How do final_result shares vary with num_of_prev_attempts and highest_education?

**Expectation:** No directional association is assumed. Small cells (n < 100) require cautious interpretation.

**Grain:** One row per dimension, category, and final_result.

**Source:** staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Shares use the category total as the denominator. This is a student-presentation grain: repeat students appear once per presentation, so rows are not independent people. Association, not causation.

**Execution note:** Inspect category denominators and flag small cells before interpreting outcome shares.

In [20]:
an3_07 = execute_query(
    "AN3.07",
    requires="AN2_10_REVIEWED",
)

AN3.07: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.07_outcome_rates_by_attempts_education.sql
Result sets returned: 1
Batch elapsed seconds: 0.086
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 36 row(s) displayed; complete display


,dimension_name,category_value,final_result,rows_in_cell,category_rows,pct_within_category,small_category_flag
0,highest_education,A Level or Equivalent,Distinction,1496,14045,10.65,False
1,highest_education,A Level or Equivalent,Fail,2707,14045,19.27,False
2,highest_education,A Level or Equivalent,Pass,5812,14045,41.38,False
3,highest_education,A Level or Equivalent,Withdrawn,4030,14045,28.69,False
4,highest_education,HE Qualification,Distinction,697,4730,14.74,False
5,highest_education,HE Qualification,Fail,790,4730,16.70,False
6,highest_education,HE Qualification,Pass,1960,4730,41.44,False
7,highest_education,HE Qualification,Withdrawn,1283,4730,27.12,False
8,highest_education,Lower Than A Level,Distinction,727,13158,5.53,False
9,highest_education,Lower Than A Level,Fail,3426,13158,26.04,False


AN3.07: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN3.07The saved result contains 36 rows across `highest_education` and `num_of_prev_attempts`, with outcome shares calculated within each dimension/category group. The SQL flag is based on the total number of rows in the category (`category_rows < 100`), not on the size of each individual `final_result` cell. Individual outcome cells can therefore be small even when the category total is at least 100.**Decision:** the SQL logic is retained; only the explanatory wording is aligned to its actual behavior. Interpret the shares descriptively because students may recur across presentations.

### AN3.08 — Withdrawal rate by presentation

**Question:** Which module-presentations have unusually high or low withdrawal?

**Expectation:** No withdrawal-rate range or presentation ordering is assumed; all differences are descriptive.

**Grain:** One row per module and presentation.

**Source:** staging.student_info.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Differences between presentations can reflect cohort size and module design, not quality. Check the denominator before ranking.

**Execution note:** Use the student-presentation denominator; do not turn descriptive rates into a presentation-quality ranking.

In [21]:
an3_08 = execute_query(
    "AN3.08",
    requires="AN2_10_REVIEWED",
)

AN3.08: EXECUTION RETURNED
Query file: sql\04_analytics\AN3.08_withdrawal_rate_by_presentation.sql
Result sets returned: 1
Batch elapsed seconds: 0.044
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 22 row(s) displayed; complete display


,code_module,code_presentation,student_rows,withdrawn_rows,pct_withdrawn,pct_pass_or_distinction
0,CCC,2014B,1936,898,46.38,34.25
1,CCC,2014J,2498,1077,43.11,40.63
2,DDD,2014B,1228,490,39.90,39.01
3,FFF,2014J,2365,855,36.15,47.23
4,DDD,2014J,1803,647,35.88,43.93
5,DDD,2013J,1938,681,35.14,42.78
6,DDD,2013B,1303,432,33.15,39.14
7,BBB,2014J,2292,749,32.68,50.26
8,FFF,2014B,1500,462,30.80,43.60
9,BBB,2014B,1613,490,30.38,45.07


AN3.08: RESULT-SET COUNT CHECK PASS (1)


### WO2-C interpretation and decisionThe retained AN3 outputs describe distinct but overlapping concepts: final outcome, presence of an unregistration date, registration timing, date ordering, and recorded VLE activity. In particular, 10,156 student-presentations are labelled Withdrawn, 10,063 have an unregistration date, and nine Fail rows also have an unregistration date. AN3.04's 45 missing registration dates and AN3.05's 39 rows with a missing registration date are different query populations because their other conditions differ.The 2014J concentration among Withdrawn rows without dates and the registration-timing gradient are hypotheses for follow-up, not causal explanations. AN3.06 measures recorded VLE-row presence/timing, not learning or verified attendance.**Decision:** WO2-C is complete as a descriptive investigation. No unified withdrawal definition, inactivity threshold, or causal interpretation is approved.

In [22]:
AN3_REVIEWED = confirm_review(    "WO2-C",    ["AN3.01", "AN3.02", "AN3.03", "AN3.04", "AN3.05", "AN3.06", "AN3.07", "AN3.08"],)


WO2-C: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved WO2-C, type REVIEWED to open its dependency gate:  REVIEWED


WO2-C reviewed: True


## 7. WO2-D — Assessment structure, dates, and scores

Run AN4.01, AN4.02, AN4.04, and AN4.05 before interpreting banked status. AN4.03 is intentionally deferred until after AN4.06.

### AN4.01 — Assessment structure by presentation

**Question:** What assessments does each presentation contain, and how do weights add up?

**Expectation:** No assessment-type pattern or weight-total rule is assumed; inspect the observed structure by presentation.

**Grain:** One row per module, presentation, and assessment type.

**Source:** staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Weight totals that differ from 100 are a finding, not an error, until the dataset documentation is checked. Do not average scores across types before reading this table.

In [23]:
an4_01 = execute_query(
    "AN4.01",
    requires="AN3_REVIEWED",
)

AN4.01: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.01_assessment_structure_by_presentation.sql
Result sets returned: 1
Batch elapsed seconds: 0.014
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 57 row(s) displayed; complete display


,code_module,code_presentation,assessment_type,assessment_count,total_weight,missing_date_count,first_due_day,last_due_day
0,AAA,2013J,Exam,1,100,1,NaN,NaN
1,AAA,2013J,TMA,5,100,0,19.0,215.0
2,AAA,2014J,Exam,1,100,1,NaN,NaN
3,AAA,2014J,TMA,5,100,0,19.0,215.0
4,BBB,2013B,CMA,5,5,0,54.0,187.0
5,BBB,2013B,Exam,1,100,1,NaN,NaN
6,BBB,2013B,TMA,6,95,0,19.0,187.0
7,BBB,2013J,CMA,5,5,0,54.0,208.0
8,BBB,2013J,Exam,1,100,1,NaN,NaN
9,BBB,2013J,TMA,6,95,0,19.0,208.0


AN4.01: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.01The saved output describes assessment counts and types across module-presentations, not submission counts. It provides the structural reference for later assessment analyses; the existence of an Exam or a particular weight does not by itself establish how its score should enter a final result.**Decision:** preserve assessment-level and submission-level grains separately. Do not assume that assessment weights sum to 100 or define a final weighted score without a documented source-semantic rule.

### AN4.02 — Assessments with missing dates

**Question:** Which assessment records have no recorded due date, and what are their observed types and submission counts?

**Expectation:** No assessment type or presentation pattern is assumed. The historical count of 11 is a reconciliation reference only.

**Grain:** One row per assessment with a NULL due date.

**Source:** staging.assessments and staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Treat assessment_type as observed data, not as a precondition. Reconcile the number of assessments separately from the number of linked submission rows. Preserve any NULL assessment_type value.

In [24]:
an4_02 = execute_query(
    "AN4.02",
    requires="AN3_REVIEWED",
)

AN4.02: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.02_assessments_missing_date.sql
Result sets returned: 1
Batch elapsed seconds: 0.032
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 11 row(s) displayed; complete display


,code_module,code_presentation,id_assessment,assessment_type,weight,submission_rows,distinct_students_with_submission
0,AAA,2013J,1757,Exam,100,0,0
1,AAA,2014J,1763,Exam,100,0,0
2,BBB,2013B,14990,Exam,100,0,0
3,BBB,2013J,15002,Exam,100,0,0
4,BBB,2014B,15014,Exam,100,0,0
5,BBB,2014J,15025,Exam,100,0,0
6,CCC,2014B,24290,Exam,100,747,747
7,CCC,2014B,40087,Exam,100,0,0
8,CCC,2014J,24299,Exam,100,1168,1168
9,CCC,2014J,40088,Exam,100,0,0


AN4.02: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.02The query returned 11 assessment records without a due date. This is an assessment-grain result, not a count of student submissions. AN4.04 separately shows that three of these assessments (all Exam, in CCC 2014B, CCC 2014J, and DDD 2014J) account for 2,865 submission rows; the other eight missing-date assessments have no submission rows in that result.**Decision:** keep missing due dates explicit. Do not infer a due date, classify submissions as late, or assume that every missing-date assessment follows one universal schedule.

### AN4.04 — Submissions without a due date

**Question:** How many submission rows are linked to assessments without a recorded due date, and how are they distributed?

**Expectation:** No assessment type, module, presentation, or shared grouping pattern is assumed. The historical total of 2,865 is a reconciliation reference only.

**Grain:** One row per module, presentation, assessment type, and banked-record status; include distinct assessment count.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** Reconcile the total submission rows with AN4.02. Do not infer that all rows share one assessment type. Preserve NULL/other banked status. A missing due date prevents direct due-date timing calculation; do not substitute course end.

In [25]:
an4_04 = execute_query(
    "AN4.04",
    requires="AN3_REVIEWED",
)

AN4.04: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.04_submissions_without_due_date.sql
Result sets returned: 1
Batch elapsed seconds: 0.035
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 3 row(s) displayed; complete display


,code_module,code_presentation,assessment_type,banked_status,assessment_count,submission_rows,distinct_students,min_date_submitted,max_date_submitted,total_without_due_date
0,CCC,2014B,Exam,NOT_BANKED,1,747,747,230,266,2865
1,CCC,2014J,Exam,NOT_BANKED,1,1168,1168,243,285,2865
2,DDD,2014J,Exam,NOT_BANKED,1,950,950,242,279,2865


AN4.04: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.04Three assessments without a recorded due date account for 2,865 submission rows: CCC 2014B 747, CCC 2014J 1,168, and DDD 2014J 950. These are Exam assessments. The remaining assessments with missing due dates have no linked submission rows in this query output.**Decision:** submissions without a due-date reference cannot be classified as early, on-time, or late from this field. Keep assessment counts, submission-row counts, and distinct-student counts separate.

### AN4.05 — Score profile and NULL scores

**Question:** What is the score distribution, including NULL and zero scores, across assessment types and banked-record statuses?

**Expectation:** No score distribution or banked-status pattern is assumed. The historical 173 NULL-score rows are a reconciliation reference only.

**Grain:** Statement 1: one row per assessment type and banked status; statement 2: one row per assessment with NULL scores and banked status.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** A NULL score means not recorded, not zero. Keep zero scores separate. Preserve all banked statuses and do not exclude records before AN4.06 is interpreted.

In [26]:
an4_05 = execute_query(
    "AN4.05",
    requires="AN3_REVIEWED",
)

AN4.05: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.05_score_profile_and_null_scores.sql
Result sets returned: 2
Batch elapsed seconds: 0.333
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 5 row(s) displayed; complete display


,assessment_type,banked_status,submission_rows,null_score_rows,zero_score_rows,out_of_range_rows,scored_rows,min_score,avg_score,median_score,max_score
0,CMA,BANKED,571,0,0,0,571,3,79.96,81.0,100
1,CMA,NOT_BANKED,69956,0,141,0,69956,0,81.04,82.0,100
2,Exam,NOT_BANKED,4959,0,3,0,4959,0,65.57,67.0,100
3,TMA,BANKED,1338,1,1,0,1337,0,71.95,74.0,100
4,TMA,NOT_BANKED,97088,172,184,0,96916,0,72.57,76.0,100


Result set 2: 78 row(s) displayed; complete display


,code_module,code_presentation,id_assessment,assessment_type,banked_status,null_score_rows,distinct_students
0,BBB,2014B,15013,TMA,NOT_BANKED,7,7
1,DDD,2013B,25339,TMA,NOT_BANKED,7,7
2,FFF,2013B,34860,TMA,NOT_BANKED,6,6
3,BBB,2013B,14984,TMA,NOT_BANKED,5,5
4,BBB,2013J,14998,TMA,NOT_BANKED,5,5
...,...,...,...,...,...,...,...
73,FFF,2014B,34886,TMA,NOT_BANKED,1,1
74,FFF,2014B,34887,TMA,NOT_BANKED,1,1
75,FFF,2014J,34903,TMA,NOT_BANKED,1,1
76,GGG,2013J,37415,TMA,NOT_BANKED,1,1


AN4.05: RESULT-SET COUNT CHECK PASS (2)


### Interpretation and decision — AN4.05Across the 173,912 `student_assessment` rows, 173,739 scores are non-NULL and 173 are NULL. The 173 NULL scores occur across 78 assessments, with no more than seven per assessment; all are TMA (172 non-banked and one banked). Numeric zero is a separate observed value: 329 rows have a score of zero. NULL and zero must not be conflated.**Decision:** retain missing scores as NULL and numeric zero as a valid recorded score. The evidence does not support a claim that missing scores are concentrated in one assessment or one module-presentation.

### WO2-D review checkpoint

Reconcile assessment counts, submission counts, missing due dates, score distributions, NULL scores, and zero scores. Document the observed assessment types and banked-status representations.

**Review checkpoint:** Send all four WO2-D outputs for review. After explicit approval, run the checkpoint cell and type `REVIEWED`.


In [27]:
AN4_D_REVIEWED = confirm_review(    "WO2-D",    ["AN4.01", "AN4.02", "AN4.04", "AN4.05"],)


WO2-D: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved WO2-D, type REVIEWED to open its dependency gate:  REVIEWED


WO2-D reviewed: True


## 8. WO2-E — Banked assessments, timing, participation, and coverage

**Required order:** AN4.06 first. Review the stored banked flag before executing AN4.03 and the remaining banked-status analyses. Banked status remains a descriptive dimension; no exclusion is approved by this notebook.

### AN4.06 — Banked assessment profile (run first)

**Question:** How are student_assessment records distributed across observed is_banked values and assessment types?

**Expectation:** No prevalence, origin, validity, or timing pattern is assumed.

**Grain:** One row per raw is_banked value (including NULL) and assessment type.

**Source:** staging.student_assessment and staging.assessments.

**Safety:** READ-ONLY.

**Runtime:** Low.

**Read as:** This is a descriptive profile of the stored flag. It does not establish why a record is banked or whether it should be excluded. Review this query before AN4.03. Preserve all statuses in downstream profiles until a separate evidence-based decision is approved.

In [28]:
an4_06 = execute_query(
    "AN4.06",
    requires="AN4_D_REVIEWED",
)

AN4.06: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.06_banked_assessments_profile.sql
Result sets returned: 1
Batch elapsed seconds: 0.408
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 5 row(s) displayed; complete display


,is_banked,assessment_type,submission_rows,distinct_students,scored_submission_rows,null_score_rows,missing_submission_date_rows,min_date_submitted,max_date_submitted,avg_score
0,0,CMA,69956,14364,69956,0,0,-8,608,81.04
1,0,Exam,4959,4633,4959,0,0,229,285,65.57
2,0,TMA,97088,22622,96916,172,0,-11,274,72.57
3,1,CMA,571,248,571,0,0,-1,-1,79.96
4,1,TMA,1338,513,1337,1,0,-1,-1,71.95


AN4.06: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.06The saved profile returns five observed combinations: `is_banked = 0` for CMA, Exam, and TMA; `is_banked = 1` for CMA and TMA. No Exam banked group appears in this result. The output also shows that banked status is not a student-level partition: the same student may contribute records in both banked and non-banked categories.**Decision:** retain the source flag as recorded. The meaning of banked status and the sentinel `date_submitted = -1` remain open; neither a filtering rule nor a date interpretation is approved.

In [29]:
AN4_06_REVIEWED = confirm_review("AN4.06", ["AN4.06"])


AN4.06: outputs are present, complete, untruncated, and tied to the current canonical SQL.


After the project lead has explicitly approved AN4.06, type REVIEWED to open its dependency gate:  REVIEWED


AN4.06 reviewed: True


### AN4.03 — Submission timing versus due date

**Question:** How early or late are submissions relative to the recorded assessment due date, and how does timing vary by banked-record status?

**Expectation:** No timing distribution or banked-status prevalence is assumed.

**Grain:** One row per assessment type, banked-record status, and timing category.

**Source:** staging.student_assessment joined to staging.assessments on id_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** days_late = date_submitted - due date. Assessments with NULL due dates are profiled separately in AN4.04. Keep is_banked as a dimension, including NULL/other values. Keep missing submission dates as a separate timing category. AN4.06 must be reviewed first; do not infer that banked rows are invalid or exclude them automatically.

In [30]:
an4_03 = execute_query(
    "AN4.03",
    requires="AN4_06_REVIEWED",
)

AN4.03: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.03_submission_timing_vs_due_date.sql
Result sets returned: 1
Batch elapsed seconds: 0.141
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 17 row(s) displayed; complete display


,assessment_type,banked_status,timing_category,submission_rows,pct_within_type_and_banked_status
0,CMA,BANKED,1_MORE_THAN_14_DAYS_EARLY,571,100.00
1,CMA,NOT_BANKED,1_MORE_THAN_14_DAYS_EARLY,28408,40.61
2,CMA,NOT_BANKED,2_1_TO_14_DAYS_EARLY,7407,10.59
3,CMA,NOT_BANKED,3_ON_DUE_DAY,1150,1.64
4,CMA,NOT_BANKED,4_1_TO_7_DAYS_LATE,32849,46.96
5,CMA,NOT_BANKED,5_MORE_THAN_7_DAYS_LATE,142,0.20
6,Exam,NOT_BANKED,1_MORE_THAN_14_DAYS_EARLY,965,46.08
7,Exam,NOT_BANKED,2_1_TO_14_DAYS_EARLY,1122,53.58
8,Exam,NOT_BANKED,4_1_TO_7_DAYS_LATE,5,0.24
9,Exam,NOT_BANKED,5_MORE_THAN_7_DAYS_LATE,2,0.10


AN4.03: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.03The saved result classifies 173,912 submission rows relative to recorded assessment due dates: 49,318 after due date, 29,469 on the due date, 92,260 before the due date, and 2,865 linked to assessments without a due date. CMA contributes 32,991 of the 49,318 after-due-date rows (66.89%). The source meaning of the CMA due date is not established by this comparison.**Decision:** these are date differences, not a validated lateness KPI. Resolve CMA due-date semantics and keep missing-due-date rows separate before using the result operationally.

### AN4.07 — Assessment participation by outcome

**Question:** How many assessment submission rows are recorded per student-presentation, by final_result and banked-record status, including students with no submission row?

**Expectation:** No outcome ordering or banked-status prevalence is assumed.

**Grain:** One row per final_result, banked status (including no-submission), and submission-count bucket; student-presentation is the counting unit.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Submission counts are grouped by the assessment's own module-presentation and student. A student-presentation may appear in more than one banked-status group if its records carry more than one status. Zero-submission student-presentations are explicitly retained.

In [31]:
an4_07 = execute_query(
    "AN4.07",
    requires="AN4_06_REVIEWED",
)

AN4.07: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.07_assessment_participation_by_outcome.sql
Result sets returned: 1
Batch elapsed seconds: 0.197
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 27 row(s) displayed; complete display


,final_result,banked_status,submission_bucket,student_presentation_rows,pct_within_final_result_and_status
0,Distinction,BANKED,1_TO_3,4,22.22
1,Distinction,BANKED,4_TO_7,12,66.67
2,Distinction,BANKED,8_PLUS,2,11.11
3,Distinction,NOT_BANKED,1_TO_3,7,0.23
4,Distinction,NOT_BANKED,4_TO_7,1088,36.00
5,Distinction,NOT_BANKED,8_PLUS,1927,63.77
6,Fail,BANKED,1_TO_3,122,54.22
7,Fail,BANKED,4_TO_7,88,39.11
8,Fail,BANKED,8_PLUS,15,6.67
9,Fail,NO_SUBMISSION_RECORD,0,1284,100.00


AN4.07: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.07The output is a submission-row distribution grouped by final result and banked-status/submission bucket. It is not a mutually exclusive partition of student-presentations: a student-presentation may have both banked and non-banked submissions, and repeated assessment rows contribute to the counts. The saved totals exceed the corresponding outcome populations by 16 Distinction, 82 Fail, 132 Pass, and 33 Withdrawn rows.**Decision:** do not read these rows as student-level participation rates or sum them as an outcome partition. Any participation measure requires a defined student-presentation grain and explicit treatment of overlap.

### AN4.08 — Score by final result

**Question:** How do recorded non-NULL assessment scores vary across final_result, assessment type, and banked-record status?

**Expectation:** No monotonic score pattern is assumed.

**Grain:** One row per final_result, assessment type, and banked status.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Scores and final_result are associated and may be mechanically related, especially for exams. This is descriptive only and is not a valid feature set for predicting final_result. The distinct count is at student-presentation grain, not unique-person grain across the full dataset. Preserve all banked statuses.

In [32]:
an4_08 = execute_query(
    "AN4.08",
    requires="AN4_06_REVIEWED",
)

AN4.08: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.08_score_by_final_result.sql
Result sets returned: 1
Batch elapsed seconds: 1.391
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 20 row(s) displayed; complete display


,final_result,assessment_type,banked_status,scored_submission_rows,distinct_student_presentations,avg_score,median_score
0,Distinction,CMA,BANKED,29,10,93.79,100.0
1,Distinction,CMA,NOT_BANKED,11657,2112,91.23,94.0
2,Fail,CMA,BANKED,249,105,79.50,80.0
3,Fail,CMA,NOT_BANKED,10157,2861,71.34,74.0
4,Pass,CMA,BANKED,185,79,81.54,83.0
5,Pass,CMA,NOT_BANKED,43973,7818,81.66,81.0
6,Withdrawn,CMA,BANKED,108,54,74.62,80.0
7,Withdrawn,CMA,NOT_BANKED,4169,2186,69.50,76.0
8,Distinction,Exam,NOT_BANKED,881,881,92.76,94.0
9,Fail,Exam,NOT_BANKED,673,673,34.11,34.0


AN4.08: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.08The saved output compares non-NULL assessment scores by final-result group, assessment type, and banked status. For non-banked TMA submissions, the mean score among Withdrawn records (64.12; median 68) is higher than the corresponding Fail group (61.46; median 64). This does not mean withdrawn students performed better overall: observed submissions are a selected subset completed before withdrawal, and groups may differ in exposure and number of submissions.**Decision:** treat score-by-outcome comparisons as retrospective and selection-sensitive. Do not use these scores as predictors of the same final outcome without a leakage-safe prediction-time design.

### AN4.09 — Submissions after unregistration

**Question:** How do recorded submission dates compare with the registration record's unregistration day, by banked status?

**Expectation:** No timing prevalence is assumed.

**Grain:** One row per module, presentation, assessment type, banked status, and timing category.

**Source:** staging.student_assessment, staging.assessments, staging.student_registration.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** This is a date comparison, not proof that a submission was accepted after withdrawal. Preserve all banked statuses. Keep missing unregistration dates, missing submission dates, and missing registration matches separate. Do not report only the after-unregistration category.

In [33]:
an4_09 = execute_query(
    "AN4.09",
    requires="AN4_06_REVIEWED",
)

AN4.09: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.09_submissions_after_unregistration.sql
Result sets returned: 1
Batch elapsed seconds: 0.411
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 178 row(s) displayed; complete display


,code_module,code_presentation,assessment_type,banked_status,timing_category,submission_rows,pct_within_type_and_status
0,AAA,2013J,TMA,NOT_BANKED,1_NO_UNREGISTRATION_DATE,1535,94.00
1,AAA,2013J,TMA,NOT_BANKED,3_BEFORE_UNREGISTRATION,98,6.00
2,AAA,2014J,TMA,BANKED,1_NO_UNREGISTRATION_DATE,28,66.67
3,AAA,2014J,TMA,BANKED,3_BEFORE_UNREGISTRATION,14,33.33
4,AAA,2014J,TMA,NOT_BANKED,1_NO_UNREGISTRATION_DATE,1366,92.67
...,...,...,...,...,...,...,...
173,GGG,2014J,TMA,BANKED,1_NO_UNREGISTRATION_DATE,7,100.00
174,GGG,2014J,TMA,NOT_BANKED,1_NO_UNREGISTRATION_DATE,1565,96.66
175,GGG,2014J,TMA,NOT_BANKED,3_BEFORE_UNREGISTRATION,49,3.03
176,GGG,2014J,TMA,NOT_BANKED,4_SAME_DAY,4,0.25


AN4.09: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.09The saved query returns 178 grouped rows describing submission dates relative to recorded unregistration dates. A submission after an unregistration date is an observed temporal ordering only; it does not establish whether the submission was accepted, graded, valid, or associated with a later administrative correction. Records without an unregistration date cannot be placed in the same before/on/after comparison.**Decision:** preserve the date categories and denominator definitions. Do not automatically discard or accept post-unregistration submissions.

### AN4.10 — Weighted-score grain and coverage check

**Question:** How much assessment weight is represented by recorded submission rows per student-presentation, and how does coverage vary by banked status?

**Expectation:** No coverage level is assumed. The output tests whether a weighted score would have comparable assessment-weight coverage.

**Grain:** One row per module, presentation, and banked status; student-presentation is the unit used to calculate each student's coverage.

**Source:** staging.student_info, staging.assessments, staging.student_assessment.

**Safety:** READ-ONLY.

**Runtime:** Moderate.

**Read as:** Weight is counted once per student-assessment-status combination after collapsing repeated physical rows at that combination. This is a metric sensitivity diagnostic, not a final weighted-score definition. A student may occur in more than one status group. Do not infer score validity from coverage alone.

In [34]:
an4_10 = execute_query(
    "AN4.10",
    requires="AN4_06_REVIEWED",
)

AN4.10: EXECUTION RETURNED
Query file: sql\04_analytics\AN4.10_weighted_score_grain_check.sql
Result sets returned: 1
Batch elapsed seconds: 0.918
Statement timeout (ms): 120000
Display row cap per result set: 500
Result set 1: 38 row(s) displayed; complete display


,code_module,code_presentation,banked_status,student_presentations_with_status,student_presentation_rows,total_weight_offered,avg_submitted_weight_share_for_status,median_submitted_weight_share_for_status
0,AAA,2013J,NOT_BANKED,365,383,200,0.439,0.500
1,AAA,2014J,BANKED,18,365,200,0.192,0.200
2,AAA,2014J,NOT_BANKED,332,365,200,0.437,0.500
3,BBB,2013B,BANKED,14,1767,200,0.055,0.025
4,BBB,2013B,NOT_BANKED,1364,1767,200,0.363,0.410
5,BBB,2013J,BANKED,50,2237,200,0.210,0.215
6,BBB,2013J,NOT_BANKED,1682,2237,200,0.373,0.495
7,BBB,2014B,BANKED,39,1613,200,0.179,0.210
8,BBB,2014B,NOT_BANKED,1187,1613,200,0.365,0.410
9,BBB,2014J,NOT_BANKED,1793,2292,200,0.378,0.500


AN4.10: RESULT-SET COUNT CHECK PASS (1)


### Interpretation and decision — AN4.10The saved output has 38 presentation/status groups. Its coverage denominator is status-specific and is not the complete student-presentation population; banked and non-banked categories can overlap for the same student-presentation. All GGG rows are 0.000 because the observed CMA/TMA weights are zero, not because submission records are necessarily absent. The median submitted-weight coverage exceeds 0.5 only in DDD (0.895–1.000), while this remains a diagnostic rather than an approved score definition.**Decision:** do not interpret the output as a complete participation measure or a final weighted score. Zero weights, missing assessment records, banked status, and Exam coverage require separate semantic treatment.

## 9. WO2-F — Evidence review and closeoutThe retained notebook contains 21 canonical WO-2 SQL investigations, 28 result sets, and 857 displayed rows. All 34 code cells have saved execution counts; no saved SQL traceback or runner row-cap truncation is recorded. AN0.01 preflight and AN1.01 dependency refresh are additional to the 21 canonical queries. The saved-output review did not reconnect to PostgreSQL or rerun SQL.### Closeout decisions- Accept WO-2 execution as complete for the 21 canonical investigations and retained output review.- Preserve the grain, join, denominator, NULL, and time-basis caveats recorded below each result.- Do not infer that AN4.07–AN4.10 received separate human review checkpoints; their interpretation boundaries are recorded, but individual approval was not evidenced.- Do not approve a unified withdrawal definition, banked-record rule, final weighted score, KPI, causal claim, or predictive feature from these exploratory outputs.- Keep RAW, QUALITY, and STAGING unchanged; no persistent database object was created by WO-2.- AN5 was separately executed and closed on 2026-11-07. AN6 is the next analytical work order and must use the accepted AN5 activity definition with its own pre-execution metric contract.**Evidence boundary:** this cell documents retained execution outputs and decisions. It does not claim that SQL was rerun during this documentation update.